In [1]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia langchain_experimental

## DataSourced Retriver

In [14]:
from langchain_community.retrievers import WikipediaRetriever

In [15]:
# Initialize the retriever (optional: set language and top_k(how many document obj))
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [17]:

# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [18]:
docs

[Document(metadata={'title': 'India–Pakistan war of 1971', 'summary': "The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.\nThirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as th

In [19]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93,

## Vector Store Retriver

In [20]:
!pip install sentence-transformers

In [21]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [22]:
!pip install -U langchain-huggingface

In [23]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [24]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [25]:
# creating sample documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [26]:
vector_store=Chroma.from_documents(
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
    documents=documents,
    collection_name="my_collection"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
#create a vector store retriver
retriever= vector_store.as_retriever(search_kwargs={"k":2})

In [28]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [29]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [30]:
results = vector_store.similarity_search(query, k=2)

## Maximal Marginal Relevance (MMR)

Maximal Marginal Relevance (MMR) is a technique designed to retrieve documents that are both relevant and diverse

“How can we pick results that are not only relevant to the query but also different from each other?”

MMR is an information retrieval algorithm designed to reduce redundancy in the retrieved results while maintaining high relevance to the query.

 Why MMR Retriever?
In regular similarity search, you may get documents that are:

All very similar to each other
Repeating the same info
Lacking diverse perspectives

MMR Retriever avoids that by:

Picking the most relevant document first
Then picking the next most relevant and least similar to already selected docs
And so on…

MMR Retriever avoids that by:

- Picking the **most relevant document** first
- Then picking the next most relevant **and least similar** to already selected docs
- And so on…

This helps especially in RAG pipelines where:

- You want your context window to contain **diverse but still relevant information**
- Especially useful when documents are semantically overlapping

In [31]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [32]:
from langchain_community.vectorstores import FAISS

In [33]:
vector_store=FAISS.from_documents(
    embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
    documents=documents,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
# Enable MMR in the retriever(making retriver object)
retriever = vector_store.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

### Understanding `lambda_mult` in MMR

`lambda_mult` is a parameter between 0 and 1 that balances search results between **relevance** to the query and **diversity** among the results.

*   **`lambda_mult` = 1**: Behaves like a normal similarity search, prioritizing **relevance**.
*   **`lambda_mult` = 0**: Prioritizes **extreme diversity**.

It should always be kept between 0 and 1. A value of **0.5** is a good starting point for a balanced approach.

In [35]:
query = "What is langchain?"
results = retriever.invoke(query)

In [36]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain helps developers build LLM applications easily.

--- Result 2 ---
Embeddings convert text into high-dimensional vectors.

--- Result 3 ---
Chroma is a vector database optimized for LLM-based search.


## Multi Query Retriver

**Multi-Query Retriever**
> Sometimes a single query might not capture all the ways information is phrased in your documents.

For example:

**Query:**
> "How can I stay healthy?"


Could mean:

- What should I eat?
- How often should I exercise?
- How can I manage stress?

A simple similarity search might **miss documents** that talk about those things but don't use the word "healthy."

---

1. **Takes your original query**
2. **Uses an LLM (e.g., GPT-3.5)** to generate multiple semantically different versions of that query
3. **Performs retrieval for each sub-query**
4. **Combines and deduplicates the results**

In [53]:
##due to conflicting version issue,not impemented here.